Step1 では入力列$U$を与えて、状態軌道$X$と総コスト$J$を順方向に計算を行った。

Step2 では、その入力列$U$をどう決めれば総コスト$J$が最小になるかを、終端から逆向きに考える。

## LQR で扱う問題

まず、離散時間状態方程式を線形として考える。

$$
x_{k+1} = A_k x_k + B_k u_k
$$

コスト$J$は2次形式で次のようなものである。ここでは最初から離散時間で考えているため、ステージコスト$\ell_k$に$\Delta t$ は掛けていない。

$$
J = \phi(x_N) + \sum_{k=0}^{N-1} \ell_k ( x_k, u_k) = \frac{1}{2} {x_N}^T Q_N {x_N} + \sum_{k=0}^{N-1}\left( \frac{1}{2} {x_k}^T Q_k {x_k} + \frac{1}{2} {u_k}^T R_k {u_k} \right)
$$

ここでは簡単のため目標状態を$x_{ref}=0$ と原点としている。
- $Q_k$ : 状態を原点へ近づける重み
- $R_k$ : 入力を小さくする重み
- $Q_N$ : 終端状態を原点へ近づける重み

$\ell_k$ と表示しているが、これは $R_k, Q_k$ のように時刻によりゲインを変更することが出来るためである。
例えば終端に近づくほど目標誤差を重視するには、以下のように設定することができる。注意として、$Q_i$は行列であるため単純な大小関係ではない。

$$
Q_0 < Q_1 < \cdots < Q_{N-1}
$$

ここではまだ 入力列 $U$ が決まっていない。

## 価値関数について

時刻 $k$ の状態 $x_k$ から終端までコストを、残りの$U_k$ について最小化する。この時の最小コストが価値関数 $V_k(x_k)$ であり、それを実現する入力列が最適入力列 $U_k^*$ である。

$U_k$は以下のように$k$から$N-1$までの入力列である。

$$
U_k = \begin{bmatrix} u_k & u_{k+1} & \cdots & u_{N-1} \end{bmatrix}^T
$$

価値関数は$J_k$を$U_k$によって最小化した後のコストである。($U_k$を変数として動かしたときに、最も小さい$J_k$の値)

$$
V_k(x_k) = \min\limits_{U_k} J_k(x_k, U_k) = \min\limits_{U_k} \left[ \sum_{i=k}^{N-1} \ell_i(x_i, u_i) + \phi(x_N) \right]
$$


一方、最小化すべき入力列は以下となる。(最小となる$J_k$の入力列$U_k$の列)

$$
U_k^* = arg \min\limits_{U_k} J_k (x_k, U_k)
$$


## Bellmanの最適性原理

時刻 $k$ から終端$N$までの最小コストは、次の2つの合計を、現在の入力 $u_k$ について最小化することで求められる。

1. 現在の入力 $u_k$ によって発生するステージコスト $\ell_k(x_k, u_k)$
1. $u_k$によって決まる次の状態 $x_{k+1}$ から先の最小コスト $V_{k+1}(x_{k+1})$

$$
V_k(x_k) = \min\limits_{u_k} \left[ \ell_k(x_k, u_k) + V_{k+1}(x_{k+1})\right]
$$

$V_{k+1}$は$k+1$から$N$までの入力について最小化済みだが、その出発点$x_{k+1}$によって値が変化する。

$$
V_{k+1}(x_{k+1}) = \min\limits_{u_{k+1}, \cdots, u_{N-1}} J_{k+1}(x_{k+1}, u_{k+1}, \cdots, u_{N-1})
$$

$x_{k+1}$は以下であるため、$u_k$によって変化する。

$$
x_{k+1} = A_k x_k + B_k u_k
$$

よって、現在の入力 $u_k$ を選ぶときには、現在のコスト $\ell_k$ と遷移先からの最小コスト $V_{k+1}$ の合計が最小になる $u_k$ を選ぶ必要がある。

$$
V_k(x_k) = \min\limits_{u_k} \left[ \ell_k(x_k, u_k) + V_{k+1}(A_k x_k + B_k u_k) \right]
$$

この式をBellman方程式と呼ぶ。

重要なことは、未来の入力列全部を一度に考えるの代わりに、今の入力$u_k$と、次の時刻以降の最小コスト $V_{k+1}$に分けていることである。

## 終端から逆向きに価値関数を計算する理由

ステージコストは$k=0$から$k=N-1$までのため、終端時刻$N$では価値関数は$x_N$のみの関数になる。

$$
V_N(x_N) = \phi(x_N) =\frac{1}{2} {x_N}^T \ Q_N \ x_N
$$

$x_N$の具体的な値が分かっているわけではなく、終端価値関数 $V_N(x_N)$ の関数形が分かっているという意味である。逆向きに計算するbackward pass では状態を変数のまま扱う。

そして、$V_N(x_N)$が分かれば、一つ前の価値関数は次で計算できる。

$$
V_{N-1}(x_{N-1}) = \min\limits_{u_{N-1}} \left[ \ell_{N-1}(x_{N-1}, u_{N-1})  + V_N(x_N) \right]
$$

これに状態方程式を代入すると以下となり、$x_N$は$x_{N-1}$と$u_{N-1}$で求まることが分かる。

$$
V_{N-1}(x_{N-1}) = \min\limits_{u_{N-1}} \left[ \ell_{N-1}(x_{N-1}, u_{N-1})  + V_N(A_{N-1} x_{N-1} + B_{N-1} u_{N-1}) \right]
$$


さらに$V_{N-1}$が分かれば、$V_{N-2}$を計算と、逆向きに価値関数を計算することができる。

$$
V_{N} \rightarrow V_{N-1} \rightarrow V_{N-2} \rightarrow \cdots \rightarrow V_{0}
$$

よって、逆向きに計算する理由は<br>

現在$k$から終端$N$までの最小コストを表す価値関数 $V_k(x_k)$ を求めるには、次の時刻$k+1$から終端$N$までの最小コスト $V_{k+1}(x_{k+1})$ が必要となる。その出発点となる終端価値関数 $V_N(x_N) = \phi(x_N)$ の関数形だけが最初から分かっているため、価値関数は終端から逆向きに計算する。